In [6]:
##### Final cleaning of data for raster model (predictors, national/sub-national totals, production)
# isolate for final vars, fix units, etc.

import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

In [7]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent.parent 

# predictor data 
predictors = pd.read_parquet(f'{cd}/Data/Clean/Training_data/raster_predictor_matrix_relative.parquet')

# country geo 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub national geo 
capital_geo = gpd.read_file(f'{cd}/Data/Clean/Geographies/subnational_capital.shp')
labor_geo = gpd.read_file(f'{cd}/Data/Clean/Geographies/subnational_labor.shp')

# save paths
save_path_predictors = f'{cd}/Data/Clean/Training_data/raster_predictor_matrix_final.parquet'
save_path_pixel_country = f'{cd}/Data/Clean/Geographies/pixel_country_crosswalk.parquet'
save_path_pixel_sub_capital = f'{cd}/Data/Clean/Geographies/pixel_sub_capital_crosswalk.parquet'
save_path_pixel_sub_labor = f'{cd}/Data/Clean/Geographies/pixel_sub_labor_crosswalk.parquet'

In [8]:
##### Clean and save predictor data 

# convert country intensities to log 
predictors['log_country_capital_intensity_USD_per_million_tonne'] = np.log1p(predictors['country_capital_intensity_USD_per_million_tonne'])
predictors['log_country_labor_intensity_jobs_per_million_tonne'] = np.log1p(predictors['country_labor_intensity_jobs_per_million_tonne'])

# Isolate final features in predictor raster to match RF models 
RF_cols = ['x', 'y', 'rtv_log_average_travel_time_port',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_oilcrops_share_base_100_production_USD',
       'rtv_log_pulses_share_base_100_production_USD',
       'rtv_log_roots_tubers_share_base_100_production_USD',
       'rtv_log_sugar_crops_share_base_100_production_USD',
       'rtv_log_ruminants_share_base_100_production_USD', 
       'log_country_capital_intensity_USD_per_million_tonne', 
       'log_country_labor_intensity_jobs_per_million_tonne']

predictors_filt = predictors[RF_cols].copy()

# save 
predictors_filt.to_parquet(save_path_predictors, index=False)

In [9]:
###### Assign each pixel to country 

# Create geometry from pixel centroids
predictors_filt['geometry'] = [Point(xy) for xy in zip(predictors_filt['x'], predictors_filt['y'])]

# Convert to GeoDataFrame (use same CRS as countries, which should be EPSG:4326)
countries = countries.to_crs("EPSG:4326")
pixel_country = gpd.GeoDataFrame(predictors_filt, geometry='geometry', crs='EPSG:4326')

# Spatial join: each pixel gets the country it falls within
pixel_country = gpd.sjoin(pixel_country, countries[['geometry', 'GID_0']], how='left', predicate='within')

# Fill missing GID_0 with nearest country
missing_mask = pixel_country['GID_0'].isna()
if missing_mask.any():
    # Reproject to equal-area for distance calculations
    pixel_country_proj = pixel_country.to_crs("EPSG:6933") # convert to equal area for projection
    countries_proj = countries.to_crs("EPSG:6933")
    
    # Find nearest country for pixels with missing GID_0
    missing_pixels = pixel_country_proj[missing_mask].copy()
    nearest = gpd.sjoin_nearest(missing_pixels[['geometry']], 
                                 countries_proj[['geometry', 'GID_0']], 
                                 how='left')
    
    # Update GID_0 with nearest country
    pixel_country.loc[missing_mask, 'GID_0'] = nearest['GID_0'].values

# clean and save 
pixel_country = pixel_country[['x', 'y', 'GID_0']]
pixel_country.to_parquet(save_path_pixel_country)

In [10]:
###### Assign each pixel to sub-national region (not all will have so no filling of missing)

### CAPITAL 
# Convert to GeoDataFrame (use same CRS as countries, which should be EPSG:4326)
capital_geo = capital_geo.to_crs("EPSG:4326")
pixel_capital_sub = gpd.GeoDataFrame(predictors_filt, geometry='geometry', crs='EPSG:4326')

# Spatial join: each pixel gets the country it falls within
pixel_capital_sub = gpd.sjoin(pixel_capital_sub, capital_geo[['geometry', 'PROJ_ID']], how='left', predicate='within')

# clean and save 
pixel_capital_sub = pixel_capital_sub[['x', 'y', 'PROJ_ID']]
pixel_capital_sub.to_parquet(save_path_pixel_sub_capital)

### LABOR 
# Convert to GeoDataFrame (use same CRS as countries, which should be EPSG:4326)
labor_geo = labor_geo.to_crs("EPSG:4326")
pixel_labor_sub = gpd.GeoDataFrame(predictors_filt, geometry='geometry', crs='EPSG:4326')

# Spatial join: each pixel gets the country it falls within
pixel_labor_sub = gpd.sjoin(pixel_labor_sub, labor_geo[['geometry', 'PROJ_ID']], how='left', predicate='within')

# clean and save 
pixel_labor_sub = pixel_labor_sub[['x', 'y', 'PROJ_ID']]
pixel_labor_sub.to_parquet(save_path_pixel_sub_labor)